In [1]:
!pip install --upgrade --force-reinstall ecos



  Using cached ecos-2.0.14-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (8.0 kB)
  Using cached numpy-2.2.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
  Using cached scipy-1.15.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached ecos-2.0.14-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (220 kB)
Using cached numpy-2.2.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.4 MB)
Using cached scipy-1.15.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (37.6 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.23.5
    Uninstalling numpy-1.23.5:
      Successfully uninstalled numpy-1.23.5
  Attempting uninstall: scipy
    Found existing installation: scipy 1.15.2
    Uninstalling scipy-1.15.2:
      Successfully uninstalled scipy-1.15.2
  Attempting uninstall: ecos
    Found existing installation: ecos 2.0.14
    Uninstalling ecos-2.0.14:
 

In [2]:
!pip freeze | grep ecos


ecos==2.0.14


In [1]:
import cvxpy as cp
print("Installed solvers:", cp.installed_solvers())


(CVXPY) Apr 04 09:12:23 PM: Encountered unexpected exception importing solver CBC:
ValueError('numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject')
(CVXPY) Apr 04 09:12:24 PM: Encountered unexpected exception importing solver CBC:
ValueError('numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject')
Installed solvers: ['CLARABEL', 'CVXOPT', 'ECOS', 'ECOS_BB', 'GLPK', 'GLPK_MI', 'HIGHS', 'OSQP', 'SCIPY', 'SCS']


Question 1 b)

In [9]:
import numpy as np
import cvxpy as cp
import csv
e_demand = np.genfromtxt("6669_Project_Electricity_Demand.csv", delimiter=',', skip_header=1)[:72, 1]
e_prices = np.genfromtxt("6669_Project_Electricity_Prices.csv", delimiter=',', skip_header=1)[:72, 2]
h_demand = np.genfromtxt("6669_Project_Hydrogen_Demand.csv", delimiter=',', skip_header=1)[:72, 1]
solar_forecast = np.genfromtxt("6669_Project_Solar_Forecast.csv", delimiter=',', skip_header=1)[:72, 1]

# Constants
num_hours = 72
solar_capacity = 2  # MW
electrolyser_power = 0.5  # MWh/hour
electrolyser_output = 9  # kg hydrogen/hour
hydrogen_price = 10  # $/kg

# Adjusted electricity demand (after solar generation)
solar_generation = solar_forecast * solar_capacity
adjusted_e_demand = np.maximum(e_demand - solar_generation, 0)

# Decision variables
E = cp.Variable(num_hours)                    # Electricity purchased
H = cp.Variable(num_hours)                    # Hydrogen purchased
Y = cp.Variable(num_hours, boolean=True)      # Electrolyser ON/OFF
H_stock = cp.Variable(num_hours + 1)          # Hydrogen stock over time

# Constraints
constraints = [H_stock[0] == 0]

for t in range(num_hours):
    # Electricity constraint
    constraints.append(E[t] >= adjusted_e_demand[t] + electrolyser_power * Y[t])

    # Hydrogen availability constraint
    constraints.append(H[t] + electrolyser_output * Y[t] + H_stock[t] >= h_demand[t])

    # Hydrogen storage update
    constraints.append(H_stock[t+1] == H[t] + electrolyser_output * Y[t] + H_stock[t] - h_demand[t])

    # Non-negativity
    constraints.append(E[t] >= 0)
    constraints.append(H[t] >= 0)
    constraints.append(H_stock[t+1] >= 0)

# Objective function
objective = cp.Minimize(cp.sum(H) * hydrogen_price + e_prices @ E)

# Solve
problem = cp.Problem(objective, constraints)
problem.solve()

# Print results
print("Solver status:", problem.status)
print("Total operational cost with electrolyser: ${:,.2f}".format(problem.value))

# Save electrolyser use to CSV
with open("Electrolyser_Use.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Hour", "ElectrolyserOn"])
    for t in range(num_hours):
        writer.writerow([t, int(round(Y[t].value))])


Solver status: optimal
Total operational cost with electrolyser: $38,546.27


In [ ]:
Question 2 a)

In [2]:
import numpy as np
import cvxpy as cp
e_demand = np.genfromtxt("6669_Project_Electricity_Demand.csv", delimiter=',', skip_header=1)[:72, 1]
e_prices = np.genfromtxt("6669_Project_Electricity_Prices.csv", delimiter=',', skip_header=1)[:72, 2]
h_demand = np.genfromtxt("6669_Project_Hydrogen_Demand.csv", delimiter=',', skip_header=1)[:72, 1]
solar_forecast = np.genfromtxt("6669_Project_Solar_Forecast.csv", delimiter=',', skip_header=1)[:72, 1]

num_hours = 72
solar_capacity = 2.0 + 0.4  # 2.4 MW
solar_generation = solar_forecast * solar_capacity  # Hourly solar generation (MWh)
E = cp.Variable(num_hours, nonneg=True)
H = cp.Variable(num_hours, nonneg=True)

#constraints
constraints = []
for t in range(num_hours):
    constraints.append(E[t] + solar_generation[t] >= e_demand[t])
    constraints.append(H[t] >= h_demand[t])
objective = cp.Minimize(e_prices @ E + 10 * cp.sum(H))
problem_q1 = cp.Problem(objective, constraints)
problem_q1.solve(solver=cp.SCS)

print("Solver status:", problem_q1.status)
print("Optimal operational cost with 2.4 MW solar capacity: ${:,.2f}".format(problem_q1.value))


Solver status: optimal
Optimal operational cost with 2.4 MW solar capacity: $44,342.06


b)

In [3]:
!pip install --upgrade --force-reinstall numpy


  Using cached numpy-2.2.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
Using cached numpy-2.2.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.4 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.4
    Uninstalling numpy-2.2.4:
      Successfully uninstalled numpy-2.2.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cylp 0.92.3 requires numpy<2.0.0,>=1.5.0, but you have numpy 2.2.4 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.2.4 which is incompatible.
tensorflow 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 2.2.4 which is incompatible.


In [4]:
!pip freeze | grep -E 'numpy|cylp|cvxpy'


cvxpy==1.6.4
cylp==0.92.3
numpy==2.2.4


In [5]:
!pip uninstall -y numpy
!pip install --no-cache-dir "numpy==1.23.5"


Found existing installation: numpy 2.2.4
Uninstalling numpy-2.2.4:
  Successfully uninstalled numpy-2.2.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 180.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
blosc2 3.2.1 requires numpy>=1.26, but you have numpy 1.23.5 which is incompatible.
jax 0.5.2 requires numpy>=1.25, but you have numpy 1.23.5 which is incompatible.
xarray 2025.1.2 requires numpy>=1.24, but you have numpy 1.23.5 which is incompatible.
albumentations 2.0.5 requires numpy>=1.24.4, but you have numpy 1.23.5 which is incompatible.
chex 0.1.89 requires numpy>=1.24.1, but you have numpy 1.23.5 which is incompatible.
scikit-image 0.25.2 requires numpy>=1.24, but you have numpy 1.23.5 which is incompatible.
albucore 0.0.23 requires numpy>=1.24.4, but you have numpy 1.23.5 which is incompatible.
bigframes 1.42.0 requires nu

In [1]:
!apt-get install -y coinor-cbc


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
coinor-cbc is already the newest version (2.10.7+ds1-1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.


In [2]:
!apt-get update
!apt-get install -y coinor-cbc

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 129 kB in 5s (28.0 kB/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Don

Question 2 b)

In [3]:
!pip install cylp
import numpy as np
import cvxpy as cp

e_demand = np.genfromtxt("6669_Project_Electricity_Demand.csv", delimiter=',', skip_header=1)[:72, 1]
e_prices = np.genfromtxt("6669_Project_Electricity_Prices.csv", delimiter=',', skip_header=1)[:72, 2]
h_demand = np.genfromtxt("6669_Project_Hydrogen_Demand.csv", delimiter=',', skip_header=1)[:72, 1]
solar_forecast = np.genfromtxt("6669_Project_Solar_Forecast.csv", delimiter=',', skip_header=1)[:72, 1]

num_hours = 72

# Parameters
base_solar_capacity = 2.0
block_capacity = 0.2
panel_cost = 900000.0
amort_period = 3.5 * 365 * 24
amort_factor = num_hours / amort_period


E = cp.Variable(num_hours, nonneg=True)
H = cp.Variable(num_hours, nonneg=True)
S = cp.Variable(integer=True)

# Constraints
constraints = []
constraints.append(S >= 0)
for t in range(num_hours):
    constraints.append(
        E[t] + (base_solar_capacity + block_capacity * S) * solar_forecast[t] >= e_demand[t]
    )
    constraints.append(H[t] >= h_demand[t])

grid_cost = e_prices @ E
hydrogen_cost = 10 * cp.sum(H)
solar_cost = panel_cost * amort_factor * S
objective = cp.Minimize(grid_cost + hydrogen_cost + solar_cost)

problem_q2 = cp.Problem(objective, constraints)
problem_q2.solve(solver=cp.CBC)

print("Solver status:", problem_q2.status)
print("Optimal operational cost over 72 hours: ${:,.2f}".format(problem_q2.value))
S_opt = S.value if S.value is not None else 0
print("Optimal number of extra 0.2 MW solar blocks (S):", int(round(S_opt)))


Solver status: optimal
Optimal operational cost over 72 hours: $44,409.00
Optimal number of extra 0.2 MW solar blocks (S): 0


Question 3

In [7]:
# Parameters
N = 72
P_H = 10  # $/kg
solar_capacity = 2  # MW
solar_generation = solar_forecast * solar_capacity
battery_capacity = 0.5  # MWh

# Decision variables
E = cp.Variable(N)  # Electricity purchased
H = cp.Variable(N)  # Hydrogen purchased
B_in = cp.Variable(N)  # Battery charging
B_out = cp.Variable(N)  # Battery discharging
B_level = cp.Variable(N+1)  # Battery state of charge

# Objective
objective = cp.Minimize(cp.sum(cp.multiply(e_prices, E) + P_H * H))

# Constraints
constraints = [
    E + solar_generation + B_out >= e_demand + B_in,
    B_level[0] == 0,
    B_level[1:] == B_level[:-1] + B_in - B_out,
    B_in <= battery_capacity,
    B_out <= battery_capacity,
    B_level <= battery_capacity,
    cp.cumsum(H) >= cp.cumsum(h_demand),
    E >= 0,
    H >= 0,
    B_in >= 0,
    B_out >= 0
]

# Solve
problem_q3 = cp.Problem(objective, constraints)
problem_q3.solve(solver=cp.CBC)

print(f"Optimal cost with battery: ${problem_q3.value:.2f}")

Optimal cost with battery: $43791.73


Question 4

In [14]:
# === Q4: Payback Periods ===

original_cost = 44409.00  # From baseline with no components

# From previous optimization problems
cost_q1 = problem.value  # From electrolyser model
cost_q2 = 44_342.06 # From optimized solar
cost_q3 = problem_q3.value  # From battery model

electrolyser_capex = 500000
battery_capex = 75000
solar_block_capex = 900000  # per 0.2 MW block
num_solar_blocks = 10       # Assume 2 MW added = 10 blocks
total_solar_capex = solar_block_capex * num_solar_blocks

# Scale savings to annual values (3-day simulation → 365 days)
scaling_factor = 365 / 3
electrolyser_savings = (original_cost - cost_q1) * scaling_factor
solar_savings = (original_cost - cost_q2) * scaling_factor
battery_savings = (original_cost - cost_q3) * scaling_factor

# Payback periods (years)
electrolyser_payback = electrolyser_capex / electrolyser_savings
solar_payback = total_solar_capex / solar_savings
battery_payback = battery_capex / battery_savings

print(f"Electrolyser payback: {electrolyser_payback:.1f} years")
print(f"Solar payback: {solar_payback:.1f} years")
print(f"Battery payback: {battery_payback:.1f} years")

# Recommendation
best = min([
    ("Electrolyser", electrolyser_payback),
    ("Solar", solar_payback),
    ("Battery", battery_payback)
], key=lambda x: x[1])

print(f"\nQ4 Recommendation: Invest in {best[0]} (Payback: {best[1]:.1f} years)")

Electrolyser payback: 0.7 years
Solar payback: 1105.1 years
Battery payback: 1.0 years

Q4 Recommendation: Invest in Electrolyser (Payback: 0.7 years)
